In [1]:
import grpc
import json
import pandas as pd

from backend.grpc.contracts import contracts_pb2, contracts_pb2_grpc

In [ ]:
{
  "preprocessing_config": {
    "dataset_preprocessing": [
      {
        "name": "PassengerId",
        "data_type": "numerical",
        "fillna_policy": "mean",
        "transformations": null,
        "drop": true
      },
      {
        "name": "Pclass",
        "data_type": "numerical",
        "fillna_policy": "mean",
        "transformations": "StandardScaler",
        "drop": false
      },
      {
        "name": "Name",
        "data_type": "categorical",
        "fillna_policy": "mode",
        "transformations": null,
        "drop": true
      },
      {
        "name": "Sex",
        "data_type": "categorical",
        "fillna_policy": "mode",
        "transformations": "LabelEncoder",
        "drop": false
      },
      {
        "name": "Age",
        "data_type": "numerical",
        "fillna_policy": "mean",
        "transformations": "StandardScaler",
        "drop": false
      },
      {
        "name": "SibSp",
        "data_type": "numerical",
        "fillna_policy": "mean",
        "transformations": "StandardScaler",
        "drop": false
      },
      {
        "name": "Parch",
        "data_type": "numerical",
        "fillna_policy": "mean",
        "transformations": "StandardScaler",
        "drop": false
      },
      {
        "name": "Ticket",
        "data_type": "categorical",
        "fillna_policy": "mode",
        "transformations": null,
        "drop": true
      },
      {
        "name": "Fare",
        "data_type": "numerical",
        "fillna_policy": "mean",
        "transformations": "MinMaxScaler",
        "drop": false
      },
      {
        "name": "Cabin",
        "data_type": "categorical",
        "fillna_policy": "mode",
        "transformations": null,
        "drop": true
      },
      {
        "name": "Embarked",
        "data_type": "categorical",
        "fillna_policy": "mode",
        "transformations": "OneHotEncoder",
        "drop": false
      },
      {
        "name": "Survived",
        "data_type": "numerical",
        "fillna_policy": "mode",
        "transformations": "StandardScaler",
        "drop": false
      }
    ],
    "target": "Survived"
  },
  "ml_config": {
    "hyperparameters": {
      "learning_rate": 0.1,
      "max_depth": 3,
      "n_estimators": 150
    },
    "model_class": "GradientBoostingRegressor"
  }
}

In [ ]:
[
    {
      "PassengerId": 1,
      "Survived": 0,
      "Pclass": 3,
      "Name": "Braund, Mr. Owen Harris",
      "Sex": "male",
      "Age": 22,
      "SibSp": 1,
      "Parch": 0,
      "Ticket": "A/5 21171",
      "Fare": 7.25,
      "Cabin": null,
      "Embarked": "S"
    },
    {
      "PassengerId": 2,
      "Survived": 1,
      "Pclass": 1,
      "Name": "Cumings, Mrs. John Bradley (Florence Briggs Thayer)",
      "Sex": "female",
      "Age": 38,
      "SibSp": 1,
      "Parch": 0,
      "Ticket": "PC 17599",
      "Fare": 71.2833,
      "Cabin": "C85",
      "Embarked": "C"
    },
    {
      "PassengerId": 3,
      "Survived": 1,
      "Pclass": 3,
      "Name": "Heikkinen, Miss. Laina",
      "Sex": "female",
      "Age": 26,
      "SibSp": 0,
      "Parch": 0,
      "Ticket": "STON/O2. 3101282",
      "Fare": 7.925,
      "Cabin": null,
      "Embarked": "S"
    },
    {
      "PassengerId": 4,
      "Survived": 1,
      "Pclass": 1,
      "Name": "Futrelle, Mrs. Jacques Heath (Lily May Peel)",
      "Sex": "female",
      "Age": 35,
      "SibSp": 1,
      "Parch": 0,
      "Ticket": "113803",
      "Fare": 53.1,
      "Cabin": "C123",
      "Embarked": "S"
    },
    {
      "PassengerId": 5,
      "Survived": 0,
      "Pclass": 3,
      "Name": "Allen, Mr. William Henry",
      "Sex": "male",
      "Age": 35,
      "SibSp": 0,
      "Parch": 0,
      "Ticket": "373450",
      "Fare": 8.05,
      "Cabin": null,
      "Embarked": "S"
    }
  ]

In [2]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:
    health_stub = contracts_pb2_grpc.HealthServiceStub(channel)
    storage_stub = contracts_pb2_grpc.DatasetRegistryServiceStub(channel)
    model_stub = contracts_pb2_grpc.ModelServiceStub(channel)
    user_stub = contracts_pb2_grpc.UserStorageServiceStub(channel)

In [3]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:
    # Инициализация всех gRPC клиентов
    health_stub = contracts_pb2_grpc.HealthServiceStub(channel)

    print("=== Проверка Health ===")
    resp = await health_stub.CheckApp(contracts_pb2.HealthRequest())
    print("Health:", resp.app)

    resp = await health_stub.CheckS3(contracts_pb2.S3HealthRequest(bucket="models-bucket"))
    print("S3:", resp.s3)

=== Проверка Health ===
Health: ok
S3: ok


In [4]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    storage_stub = contracts_pb2_grpc.DatasetRegistryServiceStub(channel)  

    user_id = "42"

    # =============================
    # 1️⃣ Загрузка CSV-файла
    # =============================
    print("\n=== Upload dataset ===")
    df = pd.DataFrame({
        "x1": [1, 2, 3, 4, 5],
        "x2": [5, 4, 3, 2, 1],
        "y": [2, 4, 6, 8, 10],
    })
    csv_bytes = df.to_csv(index=False).encode("utf-8")

    upload_resp = await storage_stub.UploadFile(
        contracts_pb2.UploadRequest(
            user_id=user_id,
            filename="train.csv",
            file_bytes=csv_bytes,
        )
    )
    print("Upload result:", upload_resp)
    data_id = upload_resp.data_id


=== Upload dataset ===
Upload result: user_id: "42"
data_id: "bd582427-6ad3-4f82-a149-351a217a7f09"
filename: "train.csv"



In [5]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    storage_stub = contracts_pb2_grpc.DatasetRegistryServiceStub(channel)  
    user_stub = contracts_pb2_grpc.UserStorageServiceStub(channel)
    
    # =============================
    # 2️⃣ Проверка использования S3
    # =============================
    print("\n=== Storage usage ===")
    usage = await storage_stub.GetUsage(contracts_pb2.UserRequest(user_id=user_id))
    print(f"Usage: {usage.usage_mb:.2f} MB")

    # =============================
    # 3️⃣ Список датасетов пользователя
    # =============================
    print("\n=== List user datasets ===")
    datasets = await user_stub.ListDatasets(contracts_pb2.UserRequest(user_id=user_id))
    for ds in datasets.datasets:
        print(f"- {ds.data_id} ({ds.name})")


=== Storage usage ===
Usage: 0.29 MB

=== List user datasets ===
- 0bf92d6b-d7af-4bd9-93d4-4de2137ea264 (train.csv)
- 1284d608-8693-4b65-9e98-c558b31bb5bc (train.csv)
- 20425558-8348-4099-9cdc-3dfca259a2a1 (train.csv)
- 91fa81f3-692d-49ac-8d6e-ad95cfe46f51 (train.csv)
- 9b78354a-2dc8-45dc-8d56-8cac656703b3 (train.csv)
- acab9abd-8364-4ff4-98db-3f67da2928cd (train.csv)
- bd582427-6ad3-4f82-a149-351a217a7f09 (train.csv)
- e2bbf641-f151-4698-8486-9c2c65a872c9 (train.csv)
- f6a950cc-17e0-454a-b13d-8c368d966a07 (train.csv)


In [6]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    model_stub = contracts_pb2_grpc.ModelServiceStub(channel)
    # =============================
    # 4️⃣ Обучение модели
    # =============================
    print("\n=== Train model ===")
    run_config = {
        "preprocessing_config": {
            "dataset_preprocessing": [
                {"name": "x1", "data_type": "numerical", "fillna_policy": "mean", "transformations": "StandardScaler"},
                {"name": "x2", "data_type": "numerical", "fillna_policy": "mean", "transformations": "StandardScaler"},
                {"name": "y",  "data_type": "numerical", "fillna_policy": "mean", "transformations": None}
            ],
            "target": "y",
        },
        "ml_config": {
            "hyperparameters": {"learning_rate": 0.1, "max_depth": 3, "n_estimators": 50},
            "model_class": "GradientBoostingRegressor",
        }
    }

    train_resp = await model_stub.Train(
        contracts_pb2.TrainRequest(
            user_id=user_id,
            data_id=data_id,
            run_config_json=json.dumps(run_config),
        )
    )
    print("Model trained:", train_resp.model_name)
    print("Metrics:", train_resp.metrics_json)



=== Train model ===
Model trained: GradientBoostingRegressor___learning_rate___0_1___max_depth___3___n_estimators___50_
Metrics: [{"name": "R2", "value": 0.9999734386011124}, {"name": "MSE", "value": 0.0002124911911006895}, {"name": "MAE", "value": 0.012369060497568007}]


In [7]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    model_stub = contracts_pb2_grpc.ModelServiceStub(channel)
    # =============================
    # 5️⃣ Инференс
    # =============================
    print("\n=== Predict ===")
    input_data = [
        {"x1": 6, "x2": 0},
        {"x1": 7, "x2": -1},
    ]

    predict_resp = await model_stub.Predict(
        contracts_pb2.PredictRequest(
            user_id=user_id,
            data_id=data_id,
            model_name=train_resp.model_name,
            input_data_json=json.dumps(input_data),
        )
    )
    print("Predictions:", predict_resp.predictions)


=== Predict ===
Predictions: [4.01030755041464, 7.98969244958536]


In [11]:
async with grpc.aio.insecure_channel("localhost:50051") as channel:  
    model_stub = contracts_pb2_grpc.ModelServiceStub(channel)
    # =============================
    # 6️⃣ Удаление модели
    # =============================
    print("\n=== Delete model ===")
    del_resp = await model_stub.Delete(
        contracts_pb2.DeleteRequest(
            user_id=user_id,
            data_id=data_id,
            model_name=train_resp.model_name,
        )
    )
    print("Delete status:", del_resp.status)


=== Delete model ===
Delete status: deleted
